# WP-01 PoC — Project Setup Verification

**Goal:** Confirm the project scaffolding is correct end-to-end:
- `catalog.csv` is readable and has the expected columns
- Each dummy row can be loaded into the `Track` Pydantic schema
- Valid rows pass without error
- An intentionally broken row raises a `ValidationError`

No audio files or external APIs are required — this notebook uses dummy data only.

In [1]:
import sys
sys.path.insert(0, '..')  # make atdj importable from notebooks/

import pandas as pd
from pydantic import ValidationError
from atdj.config import CATALOG_PATH
from atdj.schemas.track import Track

## 1. Load catalog.csv

In [2]:
df = pd.read_csv(CATALOG_PATH)
print(f"Rows: {len(df)}  |  Columns: {len(df.columns)}")
df

Rows: 5  |  Columns: 19


,id,title,orchestra,singer,style,year,decade,duration_seconds,file_path,tags,notes,bpm,key,energy,danceability,brightness,snr_estimate_db,enhanced_file_path,embedding_id
0,dummy_001,El Retirado,Di Sarli,NaN,tango,1942,1940,180.0,data/raw/dummy_001.mp3,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dummy_002,La Cumparsita,Rodriguez,NaN,tango,1944,1940,195.0,data/raw/dummy_002.mp3,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dummy_003,Desde El Alma,Canaro,NaN,vals,1940,1940,210.0,data/raw/dummy_003.mp3,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dummy_004,Milongueando En El 40,Troilo,NaN,milonga,1941,1940,175.0,data/raw/dummy_004.mp3,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,dummy_005,Cortina Sample,NaN,NaN,NaN,1950,1950,25.0,data/cortinas/dummy_005.mp3,[],cortina placeholder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Verify expected columns are present

In [3]:
required_columns = [
    "id", "title", "orchestra", "style", "year", "decade",
    "duration_seconds", "file_path"
]
missing = [c for c in required_columns if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
print("All required columns present.")

All required columns present.


## 3. Instantiate Track for each valid row

Rows with a missing `style` (like the cortina placeholder) are skipped — cortinas are handled by a separate schema.

In [4]:
tracks = []

for _, row in df.iterrows():
    if pd.isna(row.get("style")) or row["style"] == "":
        print(f"  Skipped {row['id']} — no style (cortina placeholder)")
        continue

    track = Track(
        id=row["id"],
        title=row["title"],
        orchestra=row["orchestra"],
        singer=row["singer"] if pd.notna(row.get("singer")) else None,
        style=row["style"],
        year=int(row["year"]),
        decade=int(row["decade"]),
        duration_seconds=float(row["duration_seconds"]),
        file_path=row["file_path"],
    )
    tracks.append(track)
    print(f"  OK  {track.id} | {track.title} | {track.orchestra} | {track.style} | {track.year}")

print(f"\n{len(tracks)} tracks loaded successfully.")

  OK  dummy_001 | El Retirado | Di Sarli | tango | 1942
  OK  dummy_002 | La Cumparsita | Rodriguez | tango | 1944
  OK  dummy_003 | Desde El Alma | Canaro | vals | 1940
  OK  dummy_004 | Milongueando En El 40 | Troilo | milonga | 1941
  Skipped dummy_005 — no style (cortina placeholder)

4 tracks loaded successfully.


## 4. Intentionally broken row — confirm ValidationError is raised

A negative `duration_seconds` violates the `gt=0` constraint and must be rejected.

In [5]:
try:
    bad_track = Track(
        id="bad_001",
        title="Bad Track",
        orchestra="Di Sarli",
        style="tango",
        year=1942,
        decade=1940,
        duration_seconds=-10.0,  # invalid
        file_path="data/raw/bad_001.mp3",
    )
    raise AssertionError("Expected ValidationError was not raised")
except ValidationError as e:
    print("ValidationError correctly raised:")
    for err in e.errors():
        print(f"  field='{err['loc'][0]}'  msg='{err['msg']}'")

ValidationError correctly raised:
  field='duration_seconds'  msg='Input should be greater than 0'


## 5. Summary

In [6]:
print("WP-01 PoC complete")
print(f"  catalog.csv rows loaded : {len(df)}")
print(f"  Track objects created   : {len(tracks)}")
print(f"  Validation guard        : working (bad row correctly rejected)")

WP-01 PoC complete
  catalog.csv rows loaded : 5
  Track objects created   : 4
  Validation guard        : working (bad row correctly rejected)
